# Gaming Industry Market Benchmark 2026: Exploration & Cleaning

**Dataset:** 100,000 simulated game-month records · 2,400 fictional titles · 11 genres · 6 platforms · CC BY-SA 4.0
**Goal:** audit the data, clean it with a transparent change log, then explore revenue, audience, retention, reviews and seasonality.

**Flow:** 1 Load → 2 Structure → 3 Quality audit → 4 Cleaning → 5 Feature engineering → 6 EDA → 7 Check README claims → 8 Summary

> Cleaning policy: *flag, don't silently drop.* Rows are only removed when they are exact duplicates or lack a key. Impossible values become `NaN` and are logged. Outliers are flagged, not removed.

## 0. Setup

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("data")          # folder with the Parquet/CSV tables
OUT_DIR  = Path("clean")         # cleaned output
OUT_DIR.mkdir(exist_ok=True)

# Light, technical, monospace look
plt.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110,
    "font.family": "monospace",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": .25,
    "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlesize": 11,
})
PAL = sns.color_palette("muted")
sns.set_palette(PAL)
pd.set_option("display.max_columns", 40, "display.float_format", "{:,.3f}".format)

## 1. Load

In [ ]:
def load(name):
    pq, csv = DATA_DIR / f"{name}.parquet", DATA_DIR / f"{name}.csv"
    return pd.read_parquet(pq) if pq.exists() else pd.read_csv(csv)

TABLES = ["game_performance", "genres", "platforms", "monetization_models", "genre_profile"]
raw = {t: load(t) for t in TABLES}
for t, d in raw.items():
    print(f"{t:<22} {d.shape[0]:>7,} rows x {d.shape[1]:>2} cols")

## 2. Structure

In [ ]:
gp_raw = raw["game_performance"]
display(gp_raw.head())
display(pd.DataFrame({"dtype": gp_raw.dtypes.astype(str),
                      "nulls": gp_raw.isna().sum(),
                      "n_unique": gp_raw.nunique()}))
print(f"Memory: {gp_raw.memory_usage(deep=True).sum()/1e6:.1f} MB")

**Column roles.** The README doesn't list all 15 column names, so roles are resolved by pattern. Check the printed mapping and adjust `ROLES` if something is off.

In [ ]:
def snake(c):
    c = re.sub(r"[^0-9a-zA-Z]+", "_", str(c).strip()).strip("_").lower()
    return c

ROLES = {   # role -> regex patterns, first column matching wins
    "title":        [r"^title$", r"game_?title", r"^game$", r"title"],
    "genre":        [r"^genre$"],
    "platform":     [r"^platform$"],
    "monetization": [r"^monetization$", r"monetiz"],
    "month":        [r"^month$", r"^date$", r"period", r"year_?month", r"month"],
    "mau":          [r"^monthly_active_users", r"^mau"],
    "ccu":          [r"peak_?(ccu|concurrent)", r"ccu"],
    "paying":       [r"paying"],
    "revenue":      [r"^revenue", r"revenue"],
    "review":       [r"review"],
    "stream":       [r"stream"],
    "retention":    [r"retention", r"d30"],
    "season":       [r"^season$", r"season", r"quarter"],
}

def resolve(cols):
    out = {}
    for role, pats in ROLES.items():
        for p in pats:
            hit = next((c for c in cols if re.search(p, c)), None)
            if hit:
                out[role] = hit
                break
    return out

gp = gp_raw.rename(columns=snake).copy()
C = resolve(gp.columns)
missing_roles = set(ROLES) - set(C)
print(pd.Series(C).to_string())
if missing_roles:
    print("\nNot found:", sorted(missing_roles))

## 3. Quality audit

In [ ]:
audit = {}
audit["rows"] = len(gp)
audit["exact_duplicate_rows"] = int(gp.duplicated().sum())
if {"title", "month"} <= set(C):
    audit["duplicate_title_month_keys"] = int(gp.duplicated([C["title"], C["month"]]).sum())
for role in ["title", "genre", "platform", "monetization"]:
    if role in C:
        audit[f"unique_{role}"] = gp[C[role]].nunique()
pd.Series(audit, name="value").to_frame()

In [ ]:
# Compare against the README's stated shape
expected = {"rows": 100_000, "unique_title": 2_400, "unique_genre": 11,
            "unique_platform": 6, "unique_monetization": 4}
cmp = pd.DataFrame({"expected": expected, "actual": {k: audit.get(k) for k in expected}})
cmp["ok"] = cmp.expected == cmp.actual
cmp

In [ ]:
# Missingness and label hygiene
nulls = gp.isna().mean().mul(100).round(2).rename("pct_null").to_frame()
display(nulls[nulls.pct_null > 0] if (nulls.pct_null > 0).any() else "No missing values")

for role in ["genre", "platform", "monetization", "season"]:
    if role in C:
        print(f"\n{role}:", sorted(gp[C[role]].dropna().unique().tolist()))

In [ ]:
# Numeric range check
num_roles = [r for r in ["mau", "ccu", "paying", "revenue", "review", "stream", "retention"] if r in C]
gp[[C[r] for r in num_roles]].describe(percentiles=[.01, .5, .99]).T

## 4. Cleaning

Every step is logged to `ledger`, so the final table can be traced back to the raw one.

In [ ]:
ledger = []
def log(step, n, note=""):
    ledger.append({"step": step, "rows_affected": int(n), "note": note})

df = gp.copy()

# 4.1 Strip and normalise categorical labels
for role in ["genre", "platform", "monetization", "season", "title"]:
    if role in C:
        c = C[role]
        before = df[c].copy()
        df[c] = df[c].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
        if role in ["monetization", "season"]:
            df[c] = df[c].str.lower()
        log(f"normalise {c}", (before.astype("string") != df[c]).sum(), "strip/collapse whitespace")

# 4.2 Parse the time column
if "month" in C:
    c = C["month"]
    parsed = pd.to_datetime(df[c], errors="coerce")
    log(f"parse {c} to datetime", parsed.isna().sum() - df[c].isna().sum(), "unparseable -> NaT")
    df[c] = parsed.dt.to_period("M").dt.to_timestamp()

# 4.3 Exact duplicates
n = df.duplicated().sum()
df = df.drop_duplicates()
log("drop exact duplicates", n)

# 4.4 Duplicate (title, month) keys: keep first, log how many
if {"title", "month"} <= set(C):
    n = df.duplicated([C["title"], C["month"]]).sum()
    df = df.drop_duplicates([C["title"], C["month"]], keep="first")
    log("drop duplicate title-month keys", n, "keep first")

# 4.5 Drop rows missing a key
keys = [C[r] for r in ["title", "genre", "platform", "monetization", "month"] if r in C]
n = df[keys].isna().any(axis=1).sum()
df = df.dropna(subset=keys)
log("drop rows missing a key", n, ", ".join(keys))

In [ ]:
# 4.6 Rescale shares given as percent (0-100) to fractions (0-1)
for role in ["paying", "retention"]:
    if role in C:
        c = C[role]
        if df[c].max() > 1.5:
            df[c] = df[c] / 100
            log(f"rescale {c} to 0-1", df[c].notna().sum(), "was in percent")

# 4.7 Impossible values -> NaN (logged)
RULES = {
    "mau":       (0, None),
    "ccu":       (0, None),
    "revenue":   (0, None),
    "stream":    (0, None),
    "paying":    (0, 1),
    "retention": (0, 1),
    "review":    (0, None),   # upper bound set below once the scale is known
}
if "review" in C:
    RULES["review"] = (0, 10 if df[C["review"]].max() > 5.5 else 5) if df[C["review"]].max() <= 100 else (0, 100)

for role, (lo, hi) in RULES.items():
    if role not in C: continue
    c = C[role]
    bad = (df[c] < lo) | (df[c] > hi if hi is not None else False)
    log(f"invalid {c} -> NaN", bad.sum(), f"allowed [{lo}, {hi}]")
    df.loc[bad, c] = np.nan

# 4.8 Logical consistency: peak CCU cannot exceed MAU (same units assumed)
if {"mau", "ccu"} <= set(C):
    bad = df[C["ccu"]] > df[C["mau"]]
    log("ccu > mau (flag only)", bad.sum(), "units may differ; inspect before acting")
    df["flag_ccu_gt_mau"] = bad

# 4.9 Premium titles should have paying share = 100%
if {"monetization", "paying"} <= set(C):
    prem = df[C["monetization"]].str.contains("premium", na=False)
    off = prem & (df[C["paying"]].sub(1).abs() > 1e-6)
    log("premium with paying != 100% (flag only)", off.sum())
    df["flag_premium_not_100"] = off

In [ ]:
# 4.10 Outlier flags (3x IQR, per genre so large genres don't mask small ones)
def iqr_flag(s, k=3.0):
    q1, q3 = s.quantile([.25, .75])
    return (s < q1 - k*(q3-q1)) | (s > q3 + k*(q3-q1))

for role in ["mau", "revenue"]:
    if role in C and "genre" in C:
        c = C[role]
        flag = df.groupby(C["genre"])[c].transform(iqr_flag)
        df[f"outlier_{role}"] = flag
        log(f"outlier flag {c}", flag.sum(), "per-genre 3x IQR, kept")

# 4.11 Referential integrity against reference tables
ref_checks = {}
for ref, role in [("genres", "genre"), ("platforms", "platform"), ("monetization_models", "monetization")]:
    r = raw[ref].rename(columns=snake)
    key = next((c for c in r.columns if re.search(rf"^{role}|^name$|^model$", c)), r.columns[0])
    known = set(r[key].astype("string").str.strip().str.lower())
    seen = set(df[C[role]].astype("string").str.lower().dropna())
    ref_checks[role] = sorted(seen - known)
print("Labels in game_performance not found in reference tables:", ref_checks)

In [ ]:
# 4.12 Downcast to save memory
for c in [c for c in df.columns if pd.api.types.is_string_dtype(df[c]) and not isinstance(df[c].dtype, pd.CategoricalDtype)]:
    if c != C.get("title") and df[c].nunique() < 50:
        df[c] = df[c].astype("category")
for c in df.select_dtypes("float64"):
    df[c] = pd.to_numeric(df[c], downcast="float")

pd.DataFrame(ledger)

In [ ]:
print(f"Raw {len(gp_raw):,} rows -> clean {len(df):,} rows ({len(gp_raw)-len(df):,} removed)")
try:
    df.to_parquet(OUT_DIR / "game_performance_clean.parquet", index=False)
except ImportError:
    df.to_csv(OUT_DIR / "game_performance_clean.csv", index=False)
pd.DataFrame(ledger).to_csv(OUT_DIR / "cleaning_ledger.csv", index=False)

## 5. Feature engineering

In [ ]:
d = df.copy()

if {"revenue", "mau"} <= set(C):
    # revenue is in k USD and MAU in k, so the ratio is USD per active user
    d["rev_per_mau"] = d[C["revenue"]] / d[C["mau"]].replace(0, np.nan)

if {"title", "month"} <= set(C):
    first = d.groupby(C["title"], observed=True)[C["month"]].transform("min")
    d["game_age_m"] = ((d[C["month"]].dt.year - first.dt.year) * 12 + (d[C["month"]].dt.month - first.dt.month))
    d["year"] = d[C["month"]].dt.year
    d["cal_month"] = d[C["month"]].dt.month
    d["holiday_q"] = d["cal_month"].isin([10, 11, 12])

if {"stream", "mau"} <= set(C):
    d["stream_per_kmau"] = d[C["stream"]] / d[C["mau"]].replace(0, np.nan)

d[[c for c in ["rev_per_mau", "game_age_m", "year", "holiday_q", "stream_per_kmau"] if c in d]].describe().T

## 6. Exploration

### 6.1 Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, role in zip(axes.ravel(), [r for r in ["mau", "ccu", "revenue", "review", "retention", "stream"] if r in C]):
    s = d[C[role]].dropna()
    skewed = s.skew() > 2 and (s > 0).all()
    sns.histplot(np.log10(s) if skewed else s, bins=50, ax=ax, color=PAL[0])
    ax.set_title(f"{C[role]}" + (" (log10)" if skewed else ""))
    ax.set_xlabel("")
plt.tight_layout(); plt.show()

### 6.2 Revenue and audience by genre and platform

In [ ]:
g = C["genre"]; r = C["revenue"]; m = C["mau"]
by_genre = (d.groupby(g, observed=True)
              .agg(revenue=(r, "sum"), mau_median=(m, "median"), titles=(C["title"], "nunique"))
              .assign(rev_share=lambda x: x.revenue / x.revenue.sum() * 100)
              .sort_values("revenue", ascending=False))
display(by_genre.round(1))

ax = by_genre.rev_share.sort_values().plot.barh(color=PAL[0], figsize=(8, 5))
ax.set_title("Revenue share by genre (%)"); ax.set_ylabel("")
plt.show()

In [ ]:
p = C["platform"]
plat = (d.groupby(p, observed=True)
          .agg(mau_total=(m, "sum"), revenue=(r, "sum"), rev_per_mau=("rev_per_mau", "median"))
          .sort_values("mau_total", ascending=False))
display(plat.round(2))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
plat.mau_total.plot.bar(ax=ax[0], color=PAL[1]); ax[0].set_title("Total MAU by platform")
plat.rev_per_mau.plot.bar(ax=ax[1], color=PAL[2]); ax[1].set_title("Median revenue per MAU")
for a in ax: a.set_xlabel(""); a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

### 6.3 Monetization: revenue per user and reviews

In [ ]:
mo = C["monetization"]
display(d.groupby(mo, observed=True).agg(
    rows=(r, "size"), rev_per_mau_med=("rev_per_mau", "median"),
    review_mean=(C["review"], "mean"), paying_med=(C["paying"], "median")).round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=d, x=mo, y="rev_per_mau", showfliers=False, ax=ax[0]); ax[0].set_title("Revenue per MAU")
sns.boxplot(data=d, x=mo, y=C["review"], ax=ax[1]); ax[1].set_title("Review score")
plt.tight_layout(); plt.show()

### 6.4 Seasonality

In [ ]:
monthly = d.groupby("cal_month")[[r, m]].mean()
idx = monthly / monthly.mean()
ax = idx.plot(marker="o", figsize=(9, 4))
ax.axhline(1, color="grey", lw=.8, ls="--")
ax.set_title("Seasonality index by calendar month (1.0 = average)")
ax.set_xticks(range(1, 13)); ax.set_xlabel("month")
plt.show()

ts = d.groupby(C["month"])[r].sum()
ax = ts.plot(figsize=(10, 3.5), color=PAL[0]); ax.set_title("Total monthly revenue (k USD)"); ax.set_xlabel("")
plt.show()

### 6.5 Retention and game age

In [ ]:
rt = C["retention"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
curve = d.groupby("game_age_m")[rt].mean()
curve[curve.index <= curve.index.max()].plot(ax=ax[0], color=PAL[0])
ax[0].set_title("Mean 30-day retention by game age (months)")

order = d.groupby(g, observed=True)[rt].median().sort_values().index
sns.boxplot(data=d, y=g, x=rt, order=order, showfliers=False, ax=ax[1], color=PAL[2])
ax[1].set_title("Retention by genre"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()

### 6.6 Audience vs revenue, and the streamer effect

In [ ]:
sample = d.sample(min(10_000, len(d)), random_state=42)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sc = ax[0].scatter(sample[m], sample[r], c=sample[C["paying"]], s=6, alpha=.5, cmap="viridis")
ax[0].set(xscale="log", yscale="log", xlabel=m, ylabel=r, title="Audience vs revenue")
plt.colorbar(sc, ax=ax[0], label="paying share")

s = C["stream"]
sns.regplot(data=sample, x=s, y=m, scatter_kws={"s": 5, "alpha": .3}, line_kws={"color": "crimson"}, ax=ax[1])
ax[1].set(xscale="log", yscale="log", title="Streamer hours vs MAU")
plt.tight_layout(); plt.show()

In [ ]:
num = [C[x] for x in ["mau", "ccu", "paying", "revenue", "review", "stream", "retention"] if x in C] + ["game_age_m"]
corr = d[num].corr(method="spearman")
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True, cbar=False, ax=ax)
ax.set_title("Spearman correlation"); plt.show()

## 7. Do the README's sanity anchors hold?

The README says holiday months peak and review scores slightly favour premium titles.

In [ ]:
holiday = d.groupby("holiday_q")[r].mean()
print(f"Holiday-quarter revenue lift: {holiday[True] / holiday[False] - 1:+.1%}  (README: ~+35% generator effect)")

rev_by_mo = d.groupby(mo, observed=True)[C["review"]].mean().sort_values(ascending=False)
print("\nMean review by monetization:\n", rev_by_mo.round(3).to_string())
print("\nPremium ranks first on reviews:", str(rev_by_mo.index[0]).startswith("premium"))

## 8. Summary

Fill in after running on the full data:

- **Data quality:** see `clean/cleaning_ledger.csv`. Note which flags fired (`flag_ccu_gt_mau`, `flag_premium_not_100`, outlier flags).
- **Revenue concentration:** top genres by revenue share (section 6.2).
- **Monetization:** revenue per MAU and review gap between F2P and premium (6.3).
- **Seasonality:** size of the holiday lift versus the README's +35% (section 7).
- **Retention:** decay with game age and spread across genres (6.5).
- **Caveat:** the data is simulated, so findings reflect the generator's assumptions, not the real market.